
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Create views and limit table access

In this notebook you will learn how to:
* Create views
* Manage access to views
* Use dynamic view features to restrict access to columns and rows within a table

## Set Up

Run the following cells to perform some setup. In order to avoid conflicts in a shared training environment, this will create a uniquely named database exclusively for your use. This will also create an example table called **silver** within the Unity Catalog metatore.

Note: this notebook assumes a catalog named *main* in your Unity Catalog metastore. If you need to target a different catalog, edit the following notebook, **Classroom-Setup**, before proceeding.

In [0]:
%run ./Includes/Classroom-Setup-06.3

Let's examine the contents of the **silver** table.

Note: as part of the setup, a default catalog and database was selected so we only need to specify table or view names without any additional levels.

In [0]:
SELECT * FROM silver.heartrate_device

## Create gold view

With a silver table in place, let's create a view that aggregates data from silver, presenting data suitable for the gold layer of a medallion architecture.

In [0]:
CREATE OR REPLACE VIEW gold.heartrate_avgs AS (
  SELECT mrn, name, MEAN(heartrate) avg_heartrate, DATE_TRUNC("DD", time) date
  FROM silver.heartrate_device
  GROUP BY mrn, name, DATE_TRUNC("DD", time))

Let's examine the gold view.

In [0]:
SELECT * FROM gold.heartrate_avgs

## Grant access to view [optional]

With a new view in place, let's allow users in the **account users** group to query it.

Perform this section by uncommenting the code cells and running them in sequence. You will also be prompted to run some queries in Databricks SQL. To do this:

1. Open a new tab and go to Databricks SQL.
1. Create a SQL warehouse following the instructions in *Create SQL Warehouse in Unity Catalog*.
1. Prepare to enter queries as instructed below in that environment.

In [0]:
SHOW GRANT ON VIEW gold.heartrate_avgs

In [0]:
SHOW GRANT ON TABLE silver.heartrate_device

### Grant SELECT privilege on view

The first requirement is to grant the **SELECT** privilege on the view to the **account users** group.

In [0]:
GRANT SELECT ON VIEW gold.heartrate_avgs to `account users`

### Grant USAGE privilege on catalog and database

As with tables, **USAGE** privilege is also required on the catalog and database in order to query the view.

In [0]:
GRANT USAGE ON CATALOG ${DA.catalog_name} TO `account users`;
GRANT USAGE ON DATABASE gold TO `account users`

### Query view as user

With appropriate grants in place, attempt to query the view in the Databricks SQL environment.

Run the following cell to output a query statement that reads from the view. Copy and paste the output into a new query within the SQL environment, and run the query.

In [0]:
%python
print(f"SELECT * FROM {DA.catalog_name}.gold.heartrate_avgs")

Notice that the query succeeds and the output is identical to the output above, as expected.

Now replace **`gold.heartrate_avgs`** with **`silver.heartrate_device`** and re-run the query. Notice that the query now fails. This is because the user does not have **SELECT** privilege on the **`silver.heartrate_device`** table.

In [0]:
%python
print(f"SELECT * FROM {DA.catalog_name}.silver.heartrate_device ")


Recall though, that **`heartrate_avgs`** is a view that selects from **`heartrate_device`**. How then, can the query on **`heartrate_avgs`** succeed? Unity Catalog allows the query to pass because the *owner* of that view has **SELECT** privilege on **`silver.heartrate_device`**. This is an important property since it allows us to implement views that can filter or mask rows or columns of a table, without allowing direct access to the underlying table we are trying to protect. We will see this mechanism in action next.

## Dynamic views

Dynamic views allow us to configure fine-grained access control, including:
* security at the level of columns or rows.
* data masking.

Access control is acheived through the use of functions within the definition of the view. These functions include:
* **`current_user()`**: returns the current user’s email address
* **`is_account_group_member()`**: returns TRUE if the current user is a member of the specified group

Note: for legacy compatibility, there also exists the function **`is_member()`** which returns TRUE if the current user is a member of the specified workspace-level group. Avoid using this function when implementing dynamic views in Unity Catalog.

### Restrict columns
Let's apply **`is_account_group_member()`** to mask out columns containing PII for members of the **account users** group through **`CASE`** statements within the **`SELECT`**.

Note: this is a simple example to align with the setup of this training environment. In a production system the preferable method would be to restrict rows for users who are *not* members of a specific group.

In [0]:
CREATE OR REPLACE VIEW gold.heartrate_avgs AS
SELECT
  CASE WHEN
    is_account_group_member('account users') THEN 'REDACTED'
    ELSE mrn
  END AS mrn,
  CASE WHEN
    is_account_group_member('account users') THEN 'REDACTED'
    ELSE name
  END AS name,
  MEAN(heartrate) avg_heartrate,
  DATE_TRUNC("DD", time) date
  FROM silver.heartrate_device
  GROUP BY mrn, name, DATE_TRUNC("DD", time)

Now let's reissue the grant on the updated view.

In [0]:
GRANT SELECT ON VIEW gold.heartrate_avgs to `account users`

Let's query the view, which will yield unfiltered output (assuming the current user has not been added to the **analysts** group).

In [0]:
SELECT * FROM gold.heartrate_avgs

Now re-run the query you ran earlier in the Databricks SQL environment (changing **`silver`** back to **`gold_dailyavg`**). Notice that the PII is now filtered. There is no way for members of this group to gain access to the PII since it is being protected by the view, and there is no direct access to the underlying table.

### Restrict rows
Let's now apply **`is_account_group_member()`** to filter out rows. In this case, we'll create a new gold view that returns timestamp and heartrate value, restricted for members of the **analysts** group, to rows whose device id is less than 30. Row filtering can by done by applying the conditional as a **`WHERE`** clause in the **`SELECT`**.

In [0]:
-- Create the "gold_allhr" view in the "gold" schema
CREATE OR REPLACE VIEW gold.gold_allhr AS
SELECT
  mrn,
  time,
  device_id,
  heartrate
FROM silver.heartrate_device
WHERE
  CASE WHEN
    is_account_group_member('account users') THEN device_id < 30
    ELSE TRUE
  END;

In [0]:
GRANT SELECT ON VIEW gold.gold_allhr to `account users`

In [0]:
SELECT * FROM gold.gold_allhr

Now re-run the query you ran earlier in the Databricks SQL environment (changing **`gold_dailyavg`** to **`gold_allhr`**). Notice that rows whose device ID is 30 or greater are omitted from the output.

### Data masking
One final use case for dynamic views is to mask data; that is, allow a subset of data through, but transform it in a way such that the entirety of the masked field cannot be deduced.

Here we blend the approach of row and column filtering to augment our row filtering view with data masking. But rather than replacing the entire column with the string **REDACTED**, we utilize SQL string manipulation functions to display the last two digits of the **mrn**, while masking out the rest.

Depending on your needs, SQL provides a fairly comprehensive library of string manipulation functions that can be leveraged to mask data in a number of different ways; the approach shown below illustrates a simple example of this.

In [0]:
CREATE OR REPLACE VIEW gold_allhr AS
SELECT
  CASE WHEN
    is_account_group_member('account users') THEN CONCAT("******", RIGHT(mrn, 2))
    ELSE mrn
  END AS mrn,
  time,
  device_id,
  heartrate
FROM silver.heartrate_device
WHERE
  CASE WHEN
    is_account_group_member('account users') THEN device_id < 30
    ELSE TRUE
  END

In [0]:
GRANT SELECT ON VIEW gold_allhr to `account users`

In [0]:
SELECT * FROM gold_allhr

Re-run the query against **gold_allhr** one last time in the Databricks SQL environment. Notice that, in addition to some rows being filtered, the **mrn** column is masked such that only the last two digits are displayed. This provides enough information to correlate records against known patients, but in and of itself does not divulge any PII.

## Clean up
Run the following cell to remove assets that were used in this example.

In [0]:
%python
DA.cleanup()


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>